# Notebook 6: Hebbian Plasticity and Unsupervised Learning

This notebook explores how Hebbian synaptic plasticity rules supports unsupervised learning, first in the single-neuron case and then for a network. We will illustrate the central principles using the biological phenomenon of ocular dominance. The visual cortex receives input from both eyes, but most neurons are primarily driven by a single eye. Futhermore, in many species ocular dominance is organized into distinctive stripes tiling the visual cortex.

Modifying the inputs to the two eyes early in life produces dramatic changes in ocular dominance, indicating that this structure emerges as the result of unsupervised learning. We can explain both single-cell dominance and the spatial organization through Hebbian plasticity. However, in both cases we will also need to add non-Hebbian plasticity to avoid instabilities and degeneracies in the outcomes.

The exercises in this notebook are based on Chapter 8 of Dayan and Abbott (2001).

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import linalg

mpl.rcParams['image.origin'] = 'lower'
mpl.rcParams['image.aspect'] = 'auto'
mpl.rcParams['figure.dpi'] = 120

## Background: The Averaged Hebb Rule

Recall from lecture that the basic Hebb rule for a single output neuron with firing rate $v$, weight vector $\mathbf{w}$, and presynaptic input vector $\mathbf{u}$ is

$$
\tau_w \dot{\mathbf{w}} = v\mathbf{u}
$$

Because $\tau_w$ is much larger than the time constant for the neuron, we can use the steady-state firing rate $v = \mathbf{w}^\top \mathbf{u}$. Averaging over the ensemble of training inputs and substituting the expression for $v$, this becomes the **correlation-based plasticity rule**:

$$
\tau_w \dot{\mathbf{w}} = \mathbf{Q}\mathbf{w}
$$

where $\mathbf{Q} = \langle \mathbf{u}\mathbf{u}^\top \rangle$ is the $N_u \times N_u$ input correlation matrix.

This is a linear system of ordinary differential equations, so its solution can be written in terms of $N_u$ uncoupled systems:

$$
\mathbf{w}(t) = \sum_{\mu=1}^{N_u} c_\mu(t) \mathbf{e}_\mu,
$$

each of which has the solution $c_\mu(t) = c_\mu(0) \exp(\lambda_\mu t/\tau_w)$. Because $\mathbf{Q}$ is symmetric, and all the eigenvalues $\lambda_\mu$ are positive, each $c_\mu(t)$ will grow exponentially. Over time, the eigenvector with the largest value of $\lambda_\mu$ (the *principal eigenvector*) will dominate all the others and the response will be well approximated by 

$$
v \approx \mathbf{e}_1^\top \mathbf{u}
$$

Thus, $\mathbf{w}$ aligns with the direction in input space that has the most variance, much like in principal components analysis. 

### Subtractive normalization

One problem with this conclusion is that it follows from unbounded exponential growth of the weights, which is not possible in biological neural networks. One of several ways to keep weights from growing without bound is to ensure that the sum of synaptic weights $\mathbf{n}^\top \mathbf{w}$ (where $\mathbf{n}$ is a vector of $N_u$ ones) remains constant. Biologically, this can be implemented by *homeostatic synaptic scaling*. We modify the learning rule as follows:

\begin{equation}
\tau_w \dot{\mathbf{w}} = v\mathbf{u} - \frac{v \mathbf{n}^\top \mathbf{u}}{N_u} \mathbf{n}
\end{equation}

Note that this added term is the same for all weights (no dependence on $\mathbf{w}$, and $\mathbf{n}^\top \mathbf{w}$ remains constant. We also have to impose a constraint that weights cannot be negative, and there is often a upper bound on the weights as well.

---
## Part 1: Single-Neuron Ocular Dominance

### 1.1 Setup

We consider a single neuron in layer 4 of the visual cortex receiving input from two LGN afferents: one from the right eye ($u_R$) and one from the left eye ($u_L$). Thus, $\mathbf{w} = (w_R, w_L)^\top$ and $\mathbf{u} = (u_R, u_L)^\top$, and the output is

$$v = \mathbf{w}^\top \mathbf{u} = w_R u_R + w_L u_L.$$

Under normal conditions, we will assume the two eyes are statistically equivalent, so the input correlation matrix has same-eye correlation $q_S$ on the diagonal and cross-eye correlation $q_D$ on the off-diagonal:

$$
\mathbf{Q} = \langle \mathbf{u}\mathbf{u}^\top \rangle = \begin{pmatrix} q_S & q_D \\ q_D & q_S \end{pmatrix}.
$$

The eigenvectors of this matrix are:
- $\mathbf{e}_1 = (1,1)/\sqrt{2}$, corresponding to the "binocular" mode, with eigenvalue $\lambda_1 = q_S + q_D$, and
- $\mathbf{e}_2 = (1,-1)/\sqrt{2}$, corresponding to the "ocular dominance" mode, with eigenvalue $\lambda_2 = q_S - q_D$.

To track the magnitude of the weights corresponding to each eigenvector, we'll use the following combinations

$$w_+ = w_R + w_L \qquad w_- = w_R - w_L,$$

which will obey the uncoupled differential equations

$$\tau_w \dot{w}_+ = (q_S + q_D)\, w_+ \qquad \tau_w \dot{w}_- = (q_S - q_D)\, w_-.$$

So $w_+$ grows with rate $\lambda_1$ and $w_-$ grows with rate $\lambda_2$. After eye-opening, activity from the two eyes is positively correlated ($q_D > 0$), so $\lambda_1 > \lambda_2$. With the basic Hebb rule, we predict that the principal eigenvector $w_+$ will dominate, resulting in equal weights from the two eyes. This is not what we observe in the cortex, so we clearly need to modify our model. Let's get the basic model working and then add subtractive normalization.

### 1.2 Example: Basic Hebb Rule with Saturation

The cell below simulates the basic averaged Hebb rule with a saturation constraint $0 \leq w_R, w_L \leq w_{\max}$. To simplify the example, we'll implement the learning rule using a discrete update map instead of a differential equation. Instead of 

$$\tau_w \dot{\mathbf{w}} = \mathbf{Q}\mathbf{w},$$

we take discrete steps defined by the following map:

$$\mathbf{w} \mapsto \mathbf{w} + \epsilon \mathbf{Q}\mathbf{w}$$

with weights clipped to $[0, w_{\max}]$ after each step.

In [ ]:
def run_hebb(Q, w0, epsilon=0.001, n_steps=500, w_max=1.0):
    """
    Simulate the discrete averaged Hebb rule: w <- w + epsilon * Q @ w
    with saturation constraint 0 <= w <= w_max.
    
    If normalize=True, apply subtractive normalization (see Part 1.3).
    Returns an array of shape (n_steps+1, 2) containing the weight trajectory.
    """
    w = w0.copy().astype(float)
    trajectory = [w.copy()]

    for _ in range(n_steps):
        dw = epsilon * np.dot(Q, w)
        w = w + dw
        # Saturation constraint
        w = np.clip(w, 0, w_max)
        trajectory.append(w.copy())

    return np.array(trajectory)

Let's run the model with a few different initial conditions.

In [ ]:
# Parameters
q_S = 1.0    # same-eye correlation
q_D = 0.4    # cross-eye correlation 
w_max = 1.0
Q = np.array([[q_S, q_D], [q_D, q_S]])

# Eigenvectors for reference
eigvals, eigvecs = np.linalg.eigh(Q)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
print(f"Eigenvalues: lambda_1={eigvals[0]:.2f}, lambda_2={eigvals[1]:.2f}")
print(f"Principal eigenvector e_1 = {eigvecs[:,0]}")

# Initial conditions: a grid of starting points
np.random.seed(42)
initial_conditions = [
    np.array([0.06, 0.05]),
    np.array([0.06, 0.30]),
    np.array([0.30, 0.06]),
    np.array([0.30, 0.50]),
    np.array([0.50, 0.30]),
    np.array([0.81, 0.80]),
    np.array([0.20, 0.21]),
    np.array([0.12, 0.90]),
]

fig, ax = plt.subplots(figsize=(5, 5))

for w0 in initial_conditions:
    traj = run_hebb(Q, w0, epsilon=0.02, n_steps=300, w_max=w_max)
    ax.plot(traj[:, 0], traj[:, 1], 'b-', alpha=0.6, linewidth=1)
    ax.plot(traj[0, 0], traj[0, 1], 'ko', markersize=4)
    ax.plot(traj[-1, 0], traj[-1, 1], 'b^', markersize=6)

# Mark principal eigenvector direction (slope = e1[1]/e1[0])
e1 = eigvecs[:, 0]
slope_e1 = e1[1] / e1[0]
ax.axline([0, 0], slope=slope_e1, color='gray', linestyle='--', linewidth=1, label='$\\mathbf{e}_1$ direction')

ax.set_xlim(-0.1, w_max + 0.1)
ax.set_ylim(-0.1, w_max + 0.1)
ax.set_xlabel('$w_R$', fontsize=12)
ax.set_ylabel('$w_L$', fontsize=12)
ax.set_title('Basic Hebb rule with saturation\n(no normalization)', fontsize=11)
ax.set_aspect('equal')
ax.legend(fontsize=9)
plt.tight_layout()

Notice that all the the trajectories converge to the corner $(w_{\max}, w_{\max})$, corresponding to the principal eigenvector and binocular weighting. Even with saturation, this eigenvector dominates the system regardless of how strongly one eye is favored ove the other to begin with.

### Exercise 1: Subtractive Normalization

Subtractive normalization modifies the Hebb rule so that the *sum* of all weights $\mathbf{n}^\top \mathbf{w} = w_L + w_R$ is conserved at every step. The equation at the end of the previous section becomes

$$\tau_w \dot{\mathbf{w}} = \mathbf{Q}\mathbf{w} - \frac{(\mathbf{w}^\top \mathbf{Q}\mathbf{n})\mathbf{n}}{N_u}$$

This modification gives the neuron a fixed "synaptic budget". We can only increase one weight if we decrease the other. In this case, because of the symmetry of the problem, $\mathbf{e}_1 \propto \mathbf{n}$ and there can be no growth in the $\mathbf{e}_1$ direction. In contrast, because $\mathbf{e}_2 \cdot \mathbf{n} = 0$, the $\mathbf{e}_2$ direction is unaffected.

**A.** Make a copy of `run_hebb()` and modify it to include subtractive normalization. Run this model for the same set of initial conditions used in the example above. Plot the trajectories in the $(w_R, w_L)$ plane.

**B.** Separately plot $w_+$ and $w_-$ as functions of time for one representative initial condition. Verify that $w_+$ stays approximately constant while $w_-$ grows (or shrinks) until one weight hits saturation.

In [ ]:
# Exercise 1A: Plot trajectories under subtractive normalization
# Your code here


In [ ]:
# Exercise 1B: Plot w_+ and w_- vs. time for one initial condition
# Your code here


### Exercise 2: The Role of Inter-Eye Correlations

Recall the eigenvalues: $\lambda_1 = q_S + q_D$ and $\lambda_2 = q_S - q_D$. Under subtractive normalization, the $\mathbf{e}_1$ mode is eliminated, so the dynamics of $w_-$ are governed entirely by $\lambda_2 = q_S - q_D$.

**A.** Hubel and Wiesel (1965) surgically induced strabimus (when the eyes don't point in the same direction) and then measured visual responses after development. This manipulation decorrelates the inputs from the two eyes ($q_D < 0$). Using your understanding of the math, how do you predict this will affect ocular dominance? 

**B.** Verify your answer numerically: run the simulation with $q_D = 0.4$ or $q_D = -0.4$. Starting with a very small initial difference in tuning, $\mathbf{w}(0) = (0.3, 0.301)$, plot $w_-$ over time for both cases.

**C.** Hubel and Wiesel also tested the effects of *dark-rearing*, which results in minimal input to either eye. This corresponds to $q_S \approx q_D \approx 0$. What do you predict this will result in? Explain based on the math.

**D.** Verify your answer for *C* numerically.

*Exercise 2A: written response (double-click to edit)*

In [ ]:
# Exercise 2B: Simulate dynamics for q_D < 0
# Your code here


*Exercise 2C: written response (double-click to edit):*



In [ ]:
# Exercise 2B: Simulate dynamics for q_S = q_D = 0.0
# Your code here

---
## Part 2: Ocular Dominance Stripes

### 2.1 From Single Neurons to a Cortical Map

With Hebbian plasticity and subtractive normalization, each neuron either becomes right-eye or left-eye dominant depending on its initial conditions. But visual cortex in many species exhibits a structured spatial pattern, with  dominance alternating in regular stripes across the cortical surface. A collection of neurons with no recurrent connections would produce a random patchwork, not stripes.

To model stripes, we extend to $N_v$ output neurons arranged along a 1D cortical axis, connected to each other by a fixed recurrent weight matrix $\mathbf{M}$. The steady-state output of this model is

\begin{align}
\bar{\mathbf{v}} & = \mathbf{M}\mathbf{v} + \mathbf{W}\mathbf{u} \\
 &= \mathbf{K} \mathbf{W} \mathbf{u},
\end{align}

where $\mathbf{K} = (\mathbf{I} - \mathbf{M})^{-1}$. We will keep the recurrent weights $\mathbf{M}$ fixed and apply the averaged Hebb rule to the feedforward inputs $\mathbf{W}$ using the same two-input (right/left eye) correlation matrix as before. Now the update rule becomes

$$\tau_w \frac{d\mathbf{W}}{dt} = \left\langle \mathbf{v}\mathbf{u}^\top \right\rangle = \mathbf{K} \mathbf{W} \mathbf{Q}$$

The feedforward weights $\mathbf{W}$ comprise an $N_v \times N_u$ matrix. From the previous exercise, we know this can be decomposed into orthogonal $\mathbf{w}_+$ and $\mathbf{w}_-$ column vectors with $N_v$ rows. The dynamics of these sum and difference vectors are decoupled:

\begin{align}
\tau_w \frac{d\mathbf{w}_+}{dt} & = (q_S + q_D)\, \mathbf{K} \mathbf{w}_+ \\
\tau_w \frac{d\mathbf{w}_-}{dt} & = (q_S - q_D)\, \mathbf{K} \mathbf{w}_-
\end{align}

If we apply subtractive normalization as before, $\mathbf{w}_+$ will be fixed. $\mathbf{w}_-$ can grow, and its dynamics will be dominated by the principal eigenvector of $\mathbf{K}$. Each component $a$ of $\mathbf{w}_-$ corresponds to a spatial location, and its sign determines whether the neuron in that location prefers the left or the right eye. The pattern of positive and negative values in $\mathbf{w}_-$ translate into ocular dominance stripes.

### 2.2 The K matrix and its Eigenvectors

We assume the recurrent connections are **translation-invariant**: the strength of the connection between two cortical neurons depends only on the distance between them, $K_{aa'} = K(|a - a'|)$. We also impose periodic boundary conditions to avoid edge effects.

A biologically motivated form for $K$ is a difference of Gaussians, corresponding to short-range excitation minus long-range inhibition. In Fourier space, this has a bandpass shape, which favors a particular spatial frequency of activity. This ultimately determines stripe width.

With periodic boundary conditions, the eigenvectors of a translation-invariant $\mathbf{K}$ are the **discrete Fourier modes**:

$$e^\mu_a = \cos\left(\frac{2\pi \mu a}{N_v} - \phi\right)$$

and the eigenvalues are the discrete Fourier transform $\tilde{K}(\mu)$ of $K(|a - a'|)$. The principal eigenvector is the Fourier mode at frequency $\mu^*$ where $\tilde{K}$ is maximized.

In [ ]:
def make_K_dog(N, sigma_exc, sigma_inh, A_exc=1.0, A_inh=0.1):
    """
    Construct the K matrix as a difference-of-Gaussians (DoG) with periodic
    boundary conditions.

    Parameters
    ----------
    N        : number of cortical units
    sigma_exc: width of the excitatory Gaussian (cortical units)
    sigma_inh: width of the inhibitory Gaussian (cortical units)
    A_exc    : amplitude of the excitatory Gaussian
    A_inh    : amplitude of the inhibitory Gaussian
    """
    a = np.arange(N)
    dist = np.minimum(a, N - a)  # periodic distance from unit 0
    k_row = (A_exc * np.exp(-dist**2 / (2 * sigma_exc**2))
             - A_inh * np.exp(-dist**2 / (2 * sigma_inh**2)))
    # circulant() builds the matrix with k_row as the first column;
    # each subsequent column is a cyclic shift
    return linalg.circulant(k_row)


# Parameters
N_v = 128       # number of cortical units
d_cortex = 1.2 / N_v # mm per cortical unit
# Gaussian widths specified in mm, converted to cortical units
sigma_exc = 0.02 / d_cortex   # excitatory radius ≈ 0.1 mm
sigma_inh = 0.30 / d_cortex   # inhibitory radius ≈ 0.2 mm

K = make_K_dog(N_v, sigma_exc, sigma_inh)

# Eigendecomposition
eigvals_K, eigvecs_K = np.linalg.eigh(K)
principal_idx = np.argmax(eigvals_K)
e1 = eigvecs_K[:, principal_idx].real

# The first row is the circulant's generating function — its FFT gives
# the exact eigenvalues (= Fourier transform of K), with no leakage.
k_row0 = K[:, 0]
K_tilde = np.fft.rfft(k_row0).real          # eigenvalues at each frequency
freqs = np.fft.rfftfreq(N_v, d=d_cortex)    # cycles/mm, positive only

# Center row for spatial profile plot
center = N_v // 2
k_profile = K[center, :]
positions = (np.arange(N_v) - center) * d_cortex

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel A: heatmap of the full K matrix
ax = axes[0]
extent = [0, N_v * d_cortex, 0, N_v * d_cortex]
im = ax.imshow(K, extent=extent, origin='lower', cmap='RdBu_r',
               vmax=np.max(np.abs(K)), vmin=-np.max(np.abs(K)))
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xlabel("Cortical position (mm)", fontsize=11)
ax.set_ylabel("Cortical position (mm)", fontsize=11)
ax.set_title("K matrix", fontsize=11)

# Panel B: center row of K and principal eigenvector
ax = axes[1]
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.plot(positions, k_profile, 'b-', linewidth=2, label='$K$ (center row)')
e1_scaled = e1 / np.max(np.abs(e1)) * np.max(np.abs(k_profile)) * 0.8
e1_centered = np.roll(e1_scaled, center - np.argmax(np.abs(e1_scaled)))
ax.plot(positions, e1_centered, 'r--', linewidth=1.5,
        label='Principal eigenvector (scaled)')
ax.set_xlabel('Cortical distance (mm)', fontsize=11)
ax.set_ylabel('K', fontsize=11)
ax.set_title('K profile and principal eigenvector', fontsize=11)
ax.set_xlim(positions[0], positions[-1])
ax.legend(fontsize=9)

# Panel C: Fourier transform of K (FFT of first column = exact eigenvalues)
ax = axes[2]
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.plot(freqs, K_tilde, 'b-', linewidth=2)
peak_freq = freqs[np.argmax(K_tilde)]
ax.axvline(peak_freq, color='r', linestyle='--', linewidth=1.5,
           label=f'Peak: {peak_freq:.2f} cycles/mm\n(stripe width ≈ {1/peak_freq:.1f} mm)')
ax.set_xlabel('Spatial frequency (cycles/mm)', fontsize=11)
ax.set_ylabel('$\\tilde{K}$', fontsize=11)
ax.set_title('Fourier transform of K', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()

The K function has short-range excitation and longer-range inhibition. Its Fourier transform $\tilde{K}$ has a clear peak at a non-zero spatial frequency. This is the frequency that will be amplified by the Hebbian rule, and therefore the frequency of the resulting ocular dominance stripes.

Notice that the principal eigenvector (dashed red) oscillates at exactly this spatial frequency.

### Exercise 3: Simulating Stripe Development

Now we'll simulate the full dynamics of $\mathbf{w}_-$ using a discrete update rule:

$$\mathbf{w}_- \mapsto \mathbf{w}_- + \epsilon (q_S - q_D) \mathbf{K} \mathbf{w}_-$$

We apply a saturation constraint $-w_{\max} \leq [\mathbf{w}_-]_a \leq w_{\max}$. Note that we allow $w_-$ to be negative, because a negative value means left-eye dominance.

**A.** Build on the code from exercises 1 and 2 to implement the simulation. I've provided a template for the simulation function, but you will need to fill in the inner loop. After training, plot:
   1. The final $\mathbf{w}_-$ pattern across the cortex.
   2. The power spectrum of the final $\mathbf{w}_-$ pattern. Verify that the dominant spatial frequency matches the peak of $\tilde{K}$ computed in the example above.

**B.** Run the simulation 5 times with different random seeds. Do the stripes always have similar spatial frequency (count the approximate number of cycles per mm)? Do they start at the same spatial phase (i.e., in the same position)? Explain why or why not.

In [ ]:
# Exercise 3A: Run simulation and plot final pattern and power spectrum
def run_stripe_hebb(K, q_S, q_D, epsilon=0.005, n_steps=3000, w_max=1.0, seed=0):
    """
    Simulate Hebbian development of ocular dominance stripes.

    The update rule for the ocular dominance vector w_minus is:
        w_minus <- w_minus + epsilon * (q_S - q_D) * K @ w_minus
    with saturation constraint -w_max <= w_minus <= w_max.

    Parameters
    ----------
    K       : (N_v, N_v) recurrent weight matrix
    q_S     : same-eye input correlation
    q_D     : cross-eye input correlation
    epsilon : learning rate
    n_steps : number of update steps
    w_max   : saturation bound
    seed    : random seed for initial conditions

    Returns
    -------
    w_minus : (N_v,) final ocular dominance vector
    """
    rng = np.random.default_rng(seed)
    N = K.shape[0]
    # Small random initial conditions — tiny asymmetries that will be amplified
    w_minus = rng.uniform(-0.01, 0.01, size=N)

    for _ in range(n_steps):
        # Your code here: update w_minus and apply saturation
        pass

    return w_minus


In [ ]:
# Exercise 3B: Run with multiple seeds and compare
# Your code here


*Exercise 3B — written response (double-click to edit):*



### Exercise 4: What Determines Stripe Width?

The spatial frequency of the stripes is determined by the peak of $\tilde{K}$, which in turn depends on the widths of the excitatory and inhibitory Gaussians in $K$.

**A.** Using the `make_K_dog` function, construct three K matrices with `sigma_exc` set to 0.01 mm and `sigma_inh` set to 0.05, 0.1, and 0.3 mm.

For each, plot $\tilde{K}$ and identify the peak frequency. Then run the stripe simulation and plot the resulting $\mathbf{w}_-$ pattern. Describe how stripe width changes with the width of the inhibitory surround.

**B.** Now hold `sigma_inh` at 0.1 mm and set `sigma_exc` to 0.01, 0.05, and 0.1 mm. How does the stripe width depend on the width of the excitatory center? What happens when the excitatory and inhibitory Gaussians have similar widths? Is there still a clear stripe pattern? Explain by looking at $\tilde{K}$.

In [ ]:
# Exercise 4A: Vary sigma_inh
# Your code here


In [ ]:
# Exercise 4B: Vary sigma_exc
# Your code here


*Exercise 4 — written response (double-click to edit):*

